## Imports

In [1]:
import csv
import random
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import pickle

## Import

In [2]:
better_soccer_data = []
with open('Preprocessed-Soccer-Dataset.csv', newline='', encoding='utf-8') as data:
    read = csv.reader(data)
    for x in read:
        better_soccer_data.append(x)

print('Rows loaded (including header):', len(better_soccer_data))
print('Columns:', len(better_soccer_data[0]))
print('Headers:', better_soccer_data[0])

Rows loaded (including header): 42527
Columns: 37
Headers: ['Home_L3_Pts', 'Home_L3_Goals', 'Home_L3_EnemyGoals', 'Home_L3_Wins', 'Home_L3_Losses', 'Home_L3_HomeWins', 'Home_L5_Pts', 'Home_L5_Goals', 'Home_L5_EnemyGoals', 'Home_L5_Wins', 'Home_L5_Losses', 'Home_L5_HomeWins', 'Home_L10_Pts', 'Home_L10_Goals', 'Home_L10_EnemyGoals', 'Home_L10_Wins', 'Home_L10_Losses', 'Home_L10_HomeWins', 'Away_L3_Pts', 'Away_L3_Goals', 'Away_L3_EnemyGoals', 'Away_L3_Wins', 'Away_L3_Losses', 'Away_L3_HomeWins', 'Away_L5_Pts', 'Away_L5_Goals', 'Away_L5_EnemyGoals', 'Away_L5_Wins', 'Away_L5_Losses', 'Away_L5_HomeWins', 'Away_L10_Pts', 'Away_L10_Goals', 'Away_L10_EnemyGoals', 'Away_L10_Wins', 'Away_L10_Losses', 'Away_L10_HomeWins', 'Target_Did_Home_Win']


## Data Split

In [3]:
data = better_soccer_data[1:]  # skip header row

eighty = int((len(data)) * 0.8)
training_set = data[:eighty]
testing_set  = data[eighty:]

training_labels = []
for r in training_set: training_labels.append(r.pop())
testing_labels = []
for r in testing_set: testing_labels.append(r.pop())

# Convert to numbers (float)
for i in range(len(training_set)):
    for j in range(len(training_set[i])):
        training_set[i][j] = float(training_set[i][j])

for i in range(len(testing_set)):
    for j in range(len(testing_set[i])):
        testing_set[i][j] = float(testing_set[i][j])

print('Training rows:', len(training_set))
print('Testing rows: ', len(testing_set))
print('Features per row:', len(training_set[0]))

Training rows: 34020
Testing rows:  8506
Features per row: 36


## Splitting into Subsets for each History Window

In [4]:
def build_feature_subset(dataset, feature_indices):
    subset = []
    for row in dataset:
        new_row = []
        for i in feature_indices:
            new_row.append(row[i])
        subset.append(new_row)
    return subset

In [5]:
headers = better_soccer_data[0]

l3_indices = [i for i, col in enumerate(headers[:-1]) if "L3" in col]
l5_indices = [i for i, col in enumerate(headers[:-1]) if "L5" in col]
l10_indices = [i for i, col in enumerate(headers[:-1]) if "L10" in col]

print("L3 indices:", l3_indices)
print("L5 indices:", l5_indices)
print("L10 indices:", l10_indices)

# This selected feature indices corresponding to each Window Size (3, 5, 10).
# Each window size now has a subset of entire dataset. 12 Features each.
# We can now evaluate each window size as their own separate dataset.

# Code below just to show each col in each window
# print("L3 columns:", [headers[i] for i in l3_indices])
# print("L5 columns:", [headers[i] for i in l5_indices])
# print("L10 columns:", [headers[i] for i in l10_indices])

L3 indices: [0, 1, 2, 3, 4, 5, 18, 19, 20, 21, 22, 23]
L5 indices: [6, 7, 8, 9, 10, 11, 24, 25, 26, 27, 28, 29]
L10 indices: [12, 13, 14, 15, 16, 17, 30, 31, 32, 33, 34, 35]


In [6]:
training_set_L3 = build_feature_subset(training_set, l3_indices)
testing_set_L3 = build_feature_subset(testing_set, l3_indices)

training_set_L5 = build_feature_subset(training_set, l5_indices)
testing_set_L5 = build_feature_subset(testing_set, l5_indices)

training_set_L10 = build_feature_subset(training_set, l10_indices)
testing_set_L10 = build_feature_subset(testing_set, l10_indices)

## Evaluate Models using Logistic Regression, KNN, ...

In [11]:
def evaluate_models(train_X, test_X, train_y, test_y, label):
    scaler = StandardScaler()
    train_X_scaled = scaler.fit_transform(train_X)
    test_X_scaled = scaler.transform(test_X)

    # Logistic regression
    log_reg = LogisticRegression(max_iter=1000) # max_iter can prevent warnings, keep here to be safe
    log_reg.fit(train_X_scaled, train_y)
    log_reg_score = log_reg.score(test_X_scaled, test_y)

    # KNN
    knn_results = []

    for k in [5, 7, 9, 11, 13, 15, 17, 19]: # As k increases, % increases at a smaller rate. K=15 ideal?
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(train_X_scaled, train_y)
        score = knn.score(test_X_scaled, test_y)
        knn_results.append((k, score))

    # Need 2 more ML Classifiers here for assignment

    # Log Results
    print(f'{label} Logistic Regression: {round(log_reg_score * 100, 2)}%')
    print(f'{label} KNN Results:')
    for k, score in knn_results:
        print(f'    k={k}: {round(score * 100, 2)}%')

    print()
    return log_reg_score, knn_results

In [12]:
# Creating a dictionary to store results for all 3 Window sizes using 4 ML Classifiers
results = {}

# Results for Window Size 3
results['L3'] = evaluate_models(
    training_set_L3,
    testing_set_L3,
    training_labels,
    testing_labels,
    'L3'
)
# Results for Window Size 5
results['L5'] = evaluate_models(
    training_set_L5,
    testing_set_L5,
    training_labels,
    testing_labels,
    'L5'
)
# Results for Window Size 10
results['L10'] = evaluate_models(
    training_set_L10,
    testing_set_L10,
    training_labels,
    testing_labels,
    'L10'
)

L3 Logistic Regression: 58.44%
L3 KNN Results:
    k=5: 52.13%
    k=7: 52.81%
    k=9: 53.13%
    k=11: 53.75%
    k=13: 54.24%
    k=15: 54.46%
    k=17: 54.6%
    k=19: 54.69%

L5 Logistic Regression: 60.26%
L5 KNN Results:
    k=5: 54.04%
    k=7: 54.1%
    k=9: 54.63%
    k=11: 55.47%
    k=13: 55.63%
    k=15: 56.0%
    k=17: 55.81%
    k=19: 55.85%

L10 Logistic Regression: 62.04%
L10 KNN Results:
    k=5: 55.54%
    k=7: 56.11%
    k=9: 56.57%
    k=11: 57.58%
    k=13: 58.06%
    k=15: 58.48%
    k=17: 58.91%
    k=19: 58.64%

